# E2ETune Model Inference
This notebook uses the E2ETune Mistral 7B model to generate PostgreSQL database configurations.

## Instructions
1. Paste your JSON data into the three cells below (Cells 2-4)
2. Run all cells to generate 8 diverse configurations
3. Output will be saved to `generated_configs.json`

In [ ]:
# Paste your internal_metrics.json content here
internal_metrics = {
    "xact_commit": 105.0,
    "xact_rollback": 1.0,
    "blks_read": 5364583.0,
    "blks_hit": 45400562.0,
    "tup_returned": 68156888.0,
    "tup_fetched": 36324231.0,
    "tup_inserted": 0.0,
    "conflicts": 0.0,
    "tup_updated": 0.0,
    "tup_deleted": 0.0,
    "disk_read_count": 5619846.0,
    "disk_write_count": 0.0,
    "disk_read_bytes": 46037778432.0,
    "disk_write_bytes": 0.0
}

print(f"✓ Loaded {len(internal_metrics)} internal metrics")



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Paste your query_plans.json content here (just the "query_plans" array)
query_plans = [
    "Aggregate(cost=19541.8)(Gather(cost=19541.8)(Aggregate(cost=18541.6)(Nested Loop(cost=18541.2)(Hash Join(cost=18513.8)(Nested Loop(cost=18511.7)(Hash Join(cost=15253.6)(Seq Scan(cost=13685.1); Hash(cost=2.4)(Seq Scan(cost=2.4))); Index Scan(cost=0.6)); Hash(cost=1.1)(Seq Scan(cost=1.1))); Index Scan(cost=0.6)))))",
    "Aggregate(cost=3869.9)(Nested Loop(cost=3869.9)(Nested Loop(cost=3865.9)(Nested Loop(cost=3784.8)(Nested Loop(cost=3764.7)(Seq Scan(cost=2626.1); Bitmap Heap Scan(cost=1135.5)(Bitmap Index Scan(cost=6.7))); Index Scan(cost=0.5)); Index Scan(cost=0.5)); Index Scan(cost=0.5)))",
    "Aggregate(cost=17156.1)(Nested Loop(cost=17155.8)(Nested Loop(cost=16638.7)(Nested Loop(cost=16424.9)(Seq Scan(cost=2626.1); Bitmap Heap Scan(cost=1058.4)(Bitmap Index Scan(cost=6.7))); Index Scan(cost=0.5)); Index Scan(cost=2.9)))"
]

print(f"✓ Loaded {len(query_plans)} query plans")


In [ ]:
# Paste your workload_features.json content here
workload_features = {
    "size": 33,
    "read_ratio": 1.0,
    "group_by_ratio": 0.0,
    "order_by_ratio": 0.0,
    "avg_query_length": 851.9,
    "avg_joins": 0.0,
    "filter_ratio": 1.0
}

print(f"✓ Loaded {len(workload_features)} workload features")


In [ ]:
# Settings - Load E2ETune Mistral 7B model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "springhxm/E2ETune"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model with GPU support
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,  # use half precision for GPU efficiency
        device_map="auto"           # automatically put model on GPU
    )
    model.eval()
    print("E2ETune model loaded successfully.")
    print(f"Device: {next(model.parameters()).device}")
except Exception as e:
    print("Failed to load model:", repr(e))


Loading weights: 100%|██████████| 339/339 [00:26<00:00, 12.99it/s, Materializing param=model.norm.weight]                              


Model loaded on CPU.


In [ ]:
# Build prompt from the data
import json

def format_instruction(metrics, workload, plans):
    """Build prompt from internal metrics, workload features, and query plans"""
    # Format internal metrics (only non-zero values)
    metrics_str = "\n".join([f"- {k}: {v}" for k, v in metrics.items() if float(v) > 0])
    
    # Format workload features
    workload_str = "\n".join([f"- {k}: {v}" for k, v in workload.items()])
    
    # Concatenate all query plans into a single line
    plans_summary = " ".join(plans)
    
    system_msg = "You are an expert database tuning assistant. Your task is to analyze internal database metrics and select the optimal configuration buckets (percentage ranges) for various knobs to maximize performance."
    
    instruction = f"""Based on the provided internal metrics, predict the best configuration knobs. You MUST output the result as a valid JSON object. The keys should be the database configuration knobs, and the values should be the recommended percentage ranges (e.g., '10-20%'). Do not provide any explanations or additional text.

### Internal Metrics:
{metrics_str}

### Workload Features:
{workload_str}

### Query Plan Characteristics (Examples):
{plans_summary}"""
    
    return f"<s>[INST] <<SYS>>\n{system_msg}\n<</SYS>>\n\n{instruction} [/INST]"

prompt = format_instruction(internal_metrics, workload_features, query_plans)
print("✓ Prompt prepared successfully!")
print(f"  Prompt length: {len(prompt)} characters")


In [ ]:
# Knob configuration with min/max ranges
KNOB_CONFIG = {
    "max_wal_senders": {"step": 1, "type": "integer", "default": 16.0, "min": 0.0, "max": 50},
    "autovacuum_max_workers": {"step": 1, "type": "integer", "default": 3.0, "min": 1.0, "max": 200},
    "max_connections": {"step": 1, "type": "integer", "default": 200.0, "min": 50.0, "max": 3000},
    "wal_buffers": {"step": 1, "type": "integer", "default": 2048.0, "min": -1.0, "max": 131072},
    "shared_buffers": {"step": 1, "type": "integer", "default": 8192.0, "min": 16.0, "max": 4194304},
    "autovacuum_analyze_scale_factor": {"max": 100, "min": 0, "type": "float", "default": 0.1, "step": 1},
    "autovacuum_analyze_threshold": {"max": 2147483647, "min": 0, "type": "integer", "default": 50.0, "step": 50},
    "autovacuum_naptime": {"max": 2147483, "min": 1, "type": "integer", "default": 600.0, "step": 60},
    "autovacuum_vacuum_cost_delay": {"max": 100, "min": -1, "type": "integer", "default": 20.0, "step": 1},
    "autovacuum_vacuum_cost_limit": {"max": 10000, "min": -1, "type": "integer", "default": -1.0, "step": 1},
    "autovacuum_vacuum_scale_factor": {"max": 100, "min": 0, "type": "float", "default": 0.2, "step": 1},
    "autovacuum_vacuum_threshold": {"max": 2147483647, "min": 0, "type": "integer", "default": 50.0, "step": 50},
    "backend_flush_after": {"max": 256, "min": 0, "type": "integer", "default": 0.0, "step": 1},
    "bgwriter_delay": {"max": 10000, "min": 10, "type": "integer", "default": 2000.0, "step": 10},
    "bgwriter_flush_after": {"max": 256, "min": 0, "type": "integer", "default": 64.0, "step": 1},
    "bgwriter_lru_maxpages": {"max": 1000, "min": 0, "type": "integer", "default": 100.0, "step": 10},
    "bgwriter_lru_multiplier": {"max": 10, "min": 0, "type": "integer", "default": 2.0, "step": 1},
    "checkpoint_completion_target": {"max": 1, "min": 0, "type": "float", "default": 0.5, "step": 0.1},
    "checkpoint_flush_after": {"max": 256, "min": 0, "type": "integer", "default": 32.0, "step": 2},
    "checkpoint_timeout": {"max": 3600, "min": 30, "type": "integer", "default": 900.0, "step": 10},
    "commit_delay": {"max": 100000, "min": 0, "type": "integer", "default": 0.0, "step": 10},
    "commit_siblings": {"max": 1000, "min": 0, "type": "integer", "default": 5.0, "step": 5},
    "cursor_tuple_fraction": {"max": 1, "min": 0, "type": "float", "default": 0.1, "step": 0.1},
    "deadlock_timeout": {"max": 2147483647, "min": 1, "type": "integer", "default": 1000.0, "step": 10},
    "default_statistics_target": {"max": 10000, "min": 1, "type": "integer", "default": 100.0, "step": 10},
    "effective_cache_size": {"max": 2147483647, "min": 1, "type": "integer", "default": 16384.0, "step": 64},
    "effective_io_concurrency": {"max": 1000, "min": 0, "type": "integer", "default": 1.0, "step": 1},
    "from_collapse_limit": {"max": 2147483647, "min": 1, "type": "integer", "default": 8.0, "step": 8},
    "geqo_effort": {"max": 10, "min": 1, "type": "integer", "default": 5.0, "step": 1},
    "geqo_generations": {"max": 2147483647, "min": 0, "type": "integer", "default": 0.0, "step": 8},
    "geqo_pool_size": {"max": 2147483647, "min": 0, "type": "integer", "default": 0.0, "step": 8},
    "geqo_seed": {"max": 1, "min": 0, "type": "float", "default": 0.0, "step": 0.1},
    "geqo_threshold": {"max": 2147483647, "min": 2, "type": "integer", "default": 12.0, "step": 12},
    "join_collapse_limit": {"max": 2147483647, "min": 1, "type": "integer", "default": 8.0, "step": 8},
    "maintenance_work_mem": {"max": 2147483647, "min": 1024, "type": "integer", "default": 16384.0, "step": 128},
    "temp_buffers": {"max": 1073741823, "min": 100, "type": "integer", "default": 128.0, "step": 32},
    "temp_file_limit": {"max": -1, "min": -1, "type": "integer", "default": -1.0, "step": 32},
    "vacuum_cost_delay": {"max": 100, "min": 0, "type": "integer", "default": 0.0, "step": 1},
    "vacuum_cost_limit": {"max": 10000, "min": 1, "type": "integer", "default": 200.0, "step": 10},
    "vacuum_cost_page_dirty": {"max": 10000, "min": 0, "type": "integer", "default": 20.0, "step": 5},
    "vacuum_cost_page_hit": {"max": 10000, "min": 0, "type": "integer", "default": 1.0, "step": 1},
    "vacuum_cost_page_miss": {"max": 10000, "min": 0, "type": "integer", "default": 10.0, "step": 5},
    "wal_writer_delay": {"max": 10000, "min": 1, "type": "integer", "default": 200.0, "step": 5},
    "work_mem": {"max": 2147483647, "min": 64, "type": "integer", "default": 65536.0, "step": 64}
}

def extract_json(s: str):
    """Extract first JSON object from text"""
    import json, ast, re
    m = re.search(r"\{.*?\}", s, flags=re.DOTALL)
    if not m:
        return None
    block = m.group(0)
    try:
        return json.loads(block)
    except Exception:
        try:
            return ast.literal_eval(block)
        except Exception:
            return None

def parse_percentage_range(value_str):
    """Parse percentage range like '10-20%' and return midpoint"""
    if isinstance(value_str, (int, float)):
        return float(value_str)
    
    value_str = str(value_str).strip()
    # Match patterns like "10-20%" or "10-20" or "15%"
    match = re.match(r'(\d+\.?\d*)\s*-\s*(\d+\.?\d*)%?', value_str)
    if match:
        low, high = float(match.group(1)), float(match.group(2))
        return (low + high) / 2.0
    
    # Single value like "15%" or "15"
    match = re.match(r'(\d+\.?\d*)%?', value_str)
    if match:
        return float(match.group(1))
    
    return 50.0  # default to midpoint

def convert_percentage_to_value(knob, percentage, knob_config):
    """Convert percentage (0-100) to actual knob value using min/max"""
    if knob not in knob_config:
        return None
    
    config = knob_config[knob]
    min_val = config["min"]
    max_val = config["max"]
    
    # Calculate actual value
    value = min_val + (max_val - min_val) * (percentage / 100.0)
    
    # Round based on type
    if config["type"] == "integer":
        value = int(round(value))
    
    return value

def decode_prediction(pred_dict, knob_config):
    """Convert percentage ranges to actual values"""
    if not isinstance(pred_dict, dict):
        return None
    
    real_config = {}
    for knob, value_str in pred_dict.items():
        percentage = parse_percentage_range(value_str)
        real_value = convert_percentage_to_value(knob, percentage, knob_config)
        if real_value is not None:
            real_config[knob] = real_value
    
    return real_config

# Generate 8 configurations with temperature=1.0
print("=" * 80)
print("Generating 8 configurations with temperature=1.0")
print("=" * 80)

all_configs = []

for i in range(8):
    print(f"\n{'='*80}")
    print(f"Configuration {i+1}/8")
    print('='*80)
    
    # Generate with temperature=1.0 for diversity
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=512,
            temperature=1.0,
            do_sample=True,
            top_p=0.95
        )
    
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract JSON prediction
    pred_dict = extract_json(text)
    
    if pred_dict:
        print(f"\nPredicted percentage ranges:")
        for k, v in list(pred_dict.items())[:5]:  # Show first 5
            print(f"  {k}: {v}")
        if len(pred_dict) > 5:
            print(f"  ... and {len(pred_dict) - 5} more knobs")
        
        # Convert to actual values
        real_config = decode_prediction(pred_dict, KNOB_CONFIG)
        
        if real_config:
            print(f"\nDecoded configuration (first 5 knobs):")
            for k, v in list(real_config.items())[:5]:
                print(f"  {k}: {v}")
            if len(real_config) > 5:
                print(f"  ... and {len(real_config) - 5} more knobs")
            
            all_configs.append(real_config)
        else:
            print("Failed to decode configuration")
    else:
        print("Failed to extract JSON from model output")
        print(f"Raw output (truncated): {text[:300]}...")

print(f"\n{'='*80}")
print(f"Successfully generated {len(all_configs)} configurations")
print('='*80)

# Save configurations to JSON file
# Kaggle saves to /kaggle/working/, local saves to current directory
output_file = "/kaggle/working/generated_configs.json"

with open(output_file, 'w') as f:
    json.dump(all_configs, f, indent=2)

print(f"\n✓ Saved {len(all_configs)} configurations to: {output_file}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw generation (truncated):
 system
You are a database tuning assistant.
user
You are an expert PostgreSQL DBA. Analyze the workload state, features, and query plans below.
Determine the optimal database configuration knobs to maximize performance.
Output the configuration as a JSON object where values are Bin Indices (0-19).

### Workload Context (job):
**Internal Metrics (State):**
- xact_commit: 7.0
- xact_rollback: 1.0
- blks_read: 56829.0
- blks_hit: 47909.0
- tup_returned: 1402794.0
- tup_fetched: 1267391.0
- disk_read_count: 292092.0
- disk_read_bytes: 2392817664.0

**Workload Features (Stats):**
- size: 2.0
- read_ratio: 1.0
- group_by_ratio: 0.0
- order_by_ratio: 0.0
- avg_query_length: 153.0
- avg_joins: 0.0
- filter_ratio: 1.0

**Query Plan Signals (Structure):**
- plan__count__aggregate: 4
- plan__count__s ...

Predicted bins: {'shared_buffers': 1, 'work_mem': 19, 'maintenance_work_mem': 19, 'effective_cache_size': 19, 'max_connections': 12, 'wal_buffers': 17, 'checkpoint_c